In [3]:
# %pip install stopwordsiso

In [34]:
import re
import pandas as pd
from collections import Counter
import stopwordsiso as stopwords
from datasets import load_dataset

In [35]:
stop_words = stopwords.stopwords("sv")
# print(stop_words)
stop_words = set(stop_words)

In [36]:
dataset = load_dataset("dair-ai/emotion", split="train[:1000]")
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 1000
})

In [37]:
label_column="label"
# Convert to DataFrame
df = pd.DataFrame({
    'text': dataset['text'],
    'label': dataset[label_column]
})

In [38]:
def preprocess_text(text):
    """
    Preprocess the text by removing dates, numbers, and other unwanted characters.
    
    Args:
        text (str): Input text to preprocess
        
    Returns:
        str: Preprocessed text
    """
    if not isinstance(text, str):
        return []
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove dates (various formats)
    text = re.sub(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b', '', text)
    text = re.sub(r'\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b', '', text)
    
    # Remove all numbers
    text = re.sub(r'\b\d+\b', '', text)
    
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize and filter stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in stopwords]
    
    return tokens

In [39]:
# Add tokens, apply preprocessing to the text column
df['tokens'] = df['text'].apply(preprocess_text)
# Join tokens into string for each row
df['text_joined'] = df['tokens'].apply(lambda tokens: ' '.join(tokens))
df.head()

,text,label,tokens,text_joined
0,i didnt feel humiliated,0,"[feel, humiliated]",feel humiliated
1,i can go from feeling so hopeless to so damned...,0,"[feeling, hopeless, damned, hopeful, cares, aw...",feeling hopeless damned hopeful cares awake
2,im grabbing a minute to post i feel greedy wrong,3,"[grabbing, minute, post, feel, greedy, wrong]",grabbing minute post feel greedy wrong
3,i am ever feeling nostalgic about the fireplac...,2,"[feeling, nostalgic, fireplace, property]",feeling nostalgic fireplace property
4,i am feeling grouchy,3,"[feeling, grouchy]",feeling grouchy


### which words are common in each class

In [40]:
# Create a dictionary to store counters per label
label_counters = {}

# Loop through each row and update label-specific word counter
for i, row in df.iterrows():
    label = row['label']
    tokens = row['tokens']
    if label not in label_counters:
        label_counters[label] = Counter()
    label_counters[label].update(tokens)

# Get the 10 most common words for each label
for label, counter in label_counters.items():
    print(f"Top 10 words for label '{label}':")
    print(counter.most_common(10))
    print()

Top 10 words for label '0':
[('feel', 190), ('feeling', 83), ('people', 15), ('pretty', 10), ('life', 10), ('horrible', 9), ('love', 8), ('stressed', 8), ('unwelcome', 8), ('miserable', 7)]

Top 10 words for label '3':
[('feel', 114), ('feeling', 44), ('bit', 12), ('time', 9), ('cranky', 9), ('life', 8), ('love', 8), ('frustrated', 7), ('insulted', 7), ('fucked', 6)]

Top 10 words for label '2':
[('feel', 74), ('feeling', 21), ('love', 11), ('passionate', 7), ('people', 7), ('loyal', 7), ('feelings', 7), ('loving', 6), ('time', 6), ('gentle', 5)]

Top 10 words for label '5':
[('feel', 28), ('feeling', 17), ('funny', 8), ('life', 6), ('amazed', 5), ('surprised', 4), ('amazing', 4), ('strange', 4), ('curious', 4), ('impressed', 3)]

Top 10 words for label '4':
[('feel', 72), ('feeling', 36), ('bit', 9), ('nervous', 7), ('uncertain', 6), ('skeptical', 5), ('unsure', 5), ('people', 5), ('terrified', 5), ('apprehensive', 5)]

Top 10 words for label '1':
[('feel', 258), ('feeling', 95), ('li

In [41]:
# Create a list to store all top 10 words from each label
combined_top_words = []

# Loop through each label's counter and add its top 10 to the combined list
for label, counter in label_counters.items():
    top_10 = counter.most_common(10)
    combined_top_words.extend(top_10)

# Print the combined list
print("Combined top 10 words from each label:")
combined_top_words

Combined top 10 words from each label:


[('feel', 190),
 ('feeling', 83),
 ('people', 15),
 ('pretty', 10),
 ('life', 10),
 ('horrible', 9),
 ('love', 8),
 ('stressed', 8),
 ('unwelcome', 8),
 ('miserable', 7),
 ('feel', 114),
 ('feeling', 44),
 ('bit', 12),
 ('time', 9),
 ('cranky', 9),
 ('life', 8),
 ('love', 8),
 ('frustrated', 7),
 ('insulted', 7),
 ('fucked', 6),
 ('feel', 74),
 ('feeling', 21),
 ('love', 11),
 ('passionate', 7),
 ('people', 7),
 ('loyal', 7),
 ('feelings', 7),
 ('loving', 6),
 ('time', 6),
 ('gentle', 5),
 ('feel', 28),
 ('feeling', 17),
 ('funny', 8),
 ('life', 6),
 ('amazed', 5),
 ('surprised', 4),
 ('amazing', 4),
 ('strange', 4),
 ('curious', 4),
 ('impressed', 3),
 ('feel', 72),
 ('feeling', 36),
 ('bit', 9),
 ('nervous', 7),
 ('uncertain', 6),
 ('skeptical', 5),
 ('unsure', 5),
 ('people', 5),
 ('terrified', 5),
 ('apprehensive', 5),
 ('feel', 258),
 ('feeling', 95),
 ('life', 20),
 ('happy', 18),
 ('time', 17),
 ('people', 13),
 ('pretty', 13),
 ('love', 13),
 ('feels', 10),
 ('talented', 10)]

In [42]:
len(combined_top_words)

60

In [43]:
# Create a list to store unique top words
unique_top_words = []

# Set to track seen words
seen_words = set()

# Loop through each label's top 10 and add unique words
for label, counter in label_counters.items():
    top_10 = counter.most_common(10)
    for word, count in top_10:
        if word not in seen_words:
            unique_top_words.append((word, count))
            seen_words.add(word)

# Print the deduplicated list
print("Combined top words from each label (duplicates removed):")
unique_top_words

Combined top words from each label (duplicates removed):


[('feel', 190),
 ('feeling', 83),
 ('people', 15),
 ('pretty', 10),
 ('life', 10),
 ('horrible', 9),
 ('love', 8),
 ('stressed', 8),
 ('unwelcome', 8),
 ('miserable', 7),
 ('bit', 12),
 ('time', 9),
 ('cranky', 9),
 ('frustrated', 7),
 ('insulted', 7),
 ('fucked', 6),
 ('passionate', 7),
 ('loyal', 7),
 ('feelings', 7),
 ('loving', 6),
 ('gentle', 5),
 ('funny', 8),
 ('amazed', 5),
 ('surprised', 4),
 ('amazing', 4),
 ('strange', 4),
 ('curious', 4),
 ('impressed', 3),
 ('nervous', 7),
 ('uncertain', 6),
 ('skeptical', 5),
 ('unsure', 5),
 ('terrified', 5),
 ('apprehensive', 5),
 ('happy', 18),
 ('feels', 10),
 ('talented', 10)]

In [44]:
len(unique_top_words)

37

In [45]:
# Combine all label counters into one global counter
combined_counter = Counter()

for counter in label_counters.values():
    combined_counter.update(counter)

# Get the 10 most common words across the entire dataset
top_words = combined_counter.most_common(10)

# Print the result once
print("Top 10 most common words across all labels:")
top_words

Top 10 most common words across all labels:


[('feel', 736),
 ('feeling', 296),
 ('life', 49),
 ('people', 45),
 ('love', 44),
 ('time', 43),
 ('bit', 33),
 ('pretty', 32),
 ('feels', 27),
 ('day', 22)]